# Chapter 5 Exercise — Coffee Maker Architecture

Extend your Chapter 4 coffee maker model with allocation, a port-typed interface, and an interconnection diagram.

## Problem

Your model from Chapter 4 has a `Brew` action def (with `applyWater` nested inside it as a usage) and a `CoffeeMaker` part def composing a `brewUnit : BrewUnit` part. Work through these steps:

1. Reopen `CoffeeMaker`'s body (established in Chapter 1) and add a named, usage-level allocation: `allocation brewAllocation allocate brew.applyWater to brewUnit;`. `brew` is the performed-action feature `CoffeeMaker` inherits from `BrewingSystem` (`perform action brew : Brew;`); `applyWater` is `Brew`'s own nested step (Chapter 4); `brewUnit` is `CoffeeMaker`'s own composed part. Do not write `allocate Brew to BrewUnit;` — that allocates between two definitions, not two usages, and is non-conformant (mirror the nested, usage-to-usage form main Chapter 5 notebook 02 builds for `Toaster`).
2. Add `port def WaterPort` (an `out water : ISQ::MassValue[0..*]` port), a `part def WaterPump` with a `port waterOut : WaterPort`, a `part def FilterBasket` with a `port waterIn : ~WaterPort` (the conjugate), and a `part def CoffeeFlow` assembly composing `part pump : WaterPump` and `part filterUnit : FilterBasket`, joined by a named `interface waterInterface connect pump.waterOut to filterUnit.waterIn;` — mirroring `DurationPort`/`~DurationPort` and `durationInterface` from main Chapter 5 notebook 03. Name the second part `filterUnit`, not `filter`: `filter` is a reserved word in the SysML v2 grammar (used for view filter conditions), and a bare usage named `filter` is invalid syntax that some tools silently tolerate without a diagnostic. Do not declare a plain `item def Water` or a bare `flow` statement; main Chapter 5 does not use `flow` at all.
3. Use `port_type_mismatches()` to confirm `waterOut` and `waterIn` declare related types, mirroring notebook 03's own check on `durationIn`/`durationOut`.
4. Use `model.find()` to navigate to `CoffeeDemo::CoffeeFlow` and print its `kind`, then confirm `model.find()` returns `None` for an element that does not exist, mirroring notebook 01's own negative-style check.
5. Build the interconnection intent for `CoffeeDemo::CoffeeFlow` and confirm the flow
   endpoints appear as `pump.waterOut` → `filterUnit.waterIn`, then render it as an SVG.

Verify with `model.ok == True`, `port_type_mismatches(model) == []`, and `len(intent['flows']) == 1`.


In [ ]:
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# Paste your completed Ch4 model here, then reopen CoffeeMaker's body to add
# the allocation: allocation brewAllocation allocate brew.applyWater to brewUnit;
source = """
# Your solution here
"""

model = conn.load_from_content(source, strict=False)
print(f"Model ok: {model.ok}")
if not model.ok:
    print(format_diagnostics(model.diagnostics))


In [ ]:
# Step 2: add port def WaterPort, WaterPump/FilterBasket parts with matching
# (conjugated) ports, and a CoffeeFlow assembly joined by a named interface.
source_with_interface = """
# Your solution here (extend source above)
"""

model2 = conn.load_from_content(source_with_interface, strict=False)
print(f"Model with interface ok: {model2.ok}")
if not model2.ok:
    print(format_diagnostics(model2.diagnostics))


In [ ]:
# Step 3: confirm the port types on the new interface are compatible.
from toaster.query import port_type_mismatches

mismatches = port_type_mismatches(model2)
print(f"Port type mismatches: {mismatches}")


In [ ]:
# Step 4: navigate to CoffeeFlow by qualified name, then confirm find()
# returns None for an element that does not exist.
assembly = model2.find("CoffeeDemo::CoffeeFlow")
if assembly:
    print(f"CoffeeFlow: kind={assembly.kind!r}")
else:
    print("CoffeeFlow not found — check the qualified name")

missing = model2.find("CoffeeDemo::Nonexistent")
assert missing is None
print(f"missing element: {missing}")


In [ ]:
# Step 5: build interconnection intent and check flow endpoints.
import sys
sys.path.insert(0, "../../src")
from toaster.render import build_interconnection_intent, render_interconnection
from pathlib import Path
import tempfile

intent = build_interconnection_intent(model2, "CoffeeDemo::CoffeeFlow")
print(f"Parts: {[p['name'] for p in intent['parts']]}")
print(f"Flows: {intent['flows']}")

# Render to SVG
with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "coffee_flow.svg"
    render_interconnection(intent, out)
    print(f"SVG size: {out.stat().st_size} bytes")

conn.close()
